# VneuroTK 工作流：神经数据读取与访问
> 演示 `BaseData` 对两类神经数据的完整工作流：
> - **MEG**（连续信号 + trial 切片）
> - **Ephys**（spike raster / 平均发放率 / 刺激级发放率）

**工作流**：
1. 构造路径对象（`MNEPath` / `EphysPath`）
2. 懒加载读取（`vtk.read()`）
3. 配置试次结构（`configure()`）
4. 访问 Neuro 视图（`epochs` / `continuous`）
5. 保存与重加载（HDF5 往返）
6. Ephys 各数据类型示例

In [1]:
import tempfile
from pathlib import Path

import mne  # type: ignore
import numpy as np
import pandas as pd

import vneurotk as vtk
from vneurotk.datasets import sample
from vneurotk.io import EphysPath, MNEPath, VTKPath

---
## Part 1  MEG 数据（NOD-MEG）

### 1. 构造 MNE 路径

In [2]:
root = sample.data_path("nod-meg")
nod = root / "nod-meg"
SAVE_ROOT = Path(tempfile.mkdtemp()) / "mne" / "NOD-MEG"
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

sub, session, run = sample.NOD_SUBJECT, sample.NOD_SESSION, sample.NOD_RUN

mne_path = MNEPath(
    root=nod / "meg",
    subject=sub,
    session=session,
    task=sample.NOD_TASK,
    run=run,
    suffix="meg_clean",
    extension=".fif",
)
print("MEG 文件:", mne_path.fpath)

MEG 文件: /var/tmp/zzl-zgh/cache/vneurotk/vneurotk-samples/nod-meg/meg/sub-01_ses-ImageNet01_task-ImageNet_run-01_meg_clean.fif


### 2. 懒加载读取

`vtk.read()` 仅解析路径与元数据，`neuro` 数组在首次访问时才读入内存。
传入 `pre_load=True` 可立即加载。

In [3]:
data: vtk.BaseData = vtk.read(mne_path)
data  # 此时 neuro=<lazy>

Loading MNE metadata from /var/tmp/zzl-zgh/cache/vneurotk/vneurotk-samples/nod-meg/meg/sub-01_ses-ImageNet01_task-ImageNet_run-01_meg_clean.fif


MNE-like metadata loaded: 80000 timepoints, 273 channels, sfreq=250.0 Hz


Time points,80000
Channels,273
Sampling frequency,250.00 Hz
Highpass,0.10 Hz
Lowpass,100.00 Hz
Data mode,continuous
Status,Not configured
Status,Not configured


### 3. 配置试次结构

`configure()` 绑定：刺激 ID、onset 采样点、时间窗与图片数据库。

| 参数 | Shape / 类型 | 说明 |
|---|---|---|
| `vision_onsets` | `(n_trials,)` | trial 在连续信号中的采样点索引 |
| `stim_ids` | `(n_trials,)` | 每个 onset 对应的刺激 ID |
| `trial_window` | `[start, end]` | 浮点 → 秒；整数 → 采样点 |
| `vision_db` | `dict` | `{stim_id: image}`，image 可为路径、ndarray 或 PIL.Image |

In [4]:
submeta = pd.read_csv(nod / "events" / f"sub-{sub}_events.csv")
run_meta = submeta.query(f"run == {int(run)} and session == '{session}'")
stim_ids = run_meta["image_id"].values
stims = {sid: str(nod / "stimuli" / f"{sid}.JPEG") for sid in np.unique(stim_ids)}

raw = mne.io.read_raw(mne_path.fpath, preload=False, verbose=False)
vision_onsets = vtk.utils.get_event_samples(raw, event_name="stim_on")

data.configure(
    vision_onsets=vision_onsets,
    stim_ids=stim_ids,
    vision_db=stims,
    trial_window=[-0.2, 0.8],
)
data

Configured (raw): 200 trials, 200 unique stimuli


Time points,80000
Channels,273
Sampling frequency,250.00 Hz
Highpass,0.10 Hz
Lowpass,100.00 Hz
Data mode,continuous
n_visual,200
Baseline,"[-50, 0]"
Trial window,"[-50, 200]"


### 4. Neuro 视图

`data.neuro` 返回 `NeuroData`（`np.ndarray` 子类），提供两种结构化视图：

| 属性 | Shape | 说明 |
|---|---|---|
| `.epochs` | `(n_trials, n_timebins, nchan)` | 以 trial 为单位切片 |
| `.continuous` | `(total_samples, nchan)` | 所有 trial 拼接的连续信号 |

In [5]:
print(f"n_trials  = {data.n_trials}")
print(f"nchan     = {data.nchan}")
print(f"ntime     = {data.ntime}")
print(f"data_mode = {data.data_mode}")

neuro = data.neuro  # 触发懒加载，原始 shape (total_samples, nchan)
print(f"\nraw shape  : {neuro.shape}")
print(f"epochs     : {data.neuro.epochs.shape}")
print(f"continuous : {data.neuro.continuous.shape}")

Lazy-loading neuro data...


Reading MNE data into memory from /var/tmp/zzl-zgh/cache/vneurotk/vneurotk-samples/nod-meg/meg/sub-01_ses-ImageNet01_task-ImageNet_run-01_meg_clean.fif


WARNING | neuro is already in continuous format (shape (80000, 273)), returning as-is


n_trials  = 200
nchan     = 273
ntime     = 80000
data_mode = continuous

raw shape  : (80000, 273)
epochs     : (200, 250, 273)
continuous : (80000, 273)


### 5. 保存与重加载

`save()` 将神经数据、配置与图片数据库打包写入单个 **HDF5** 文件。
重加载后 `vision.db` 变为 `LazyH5Dict`，仅按需读取图片像素。

In [6]:
save_path = VTKPath(SAVE_ROOT, subject=sub, session=session, task="ImageNet", run=run)
data.save(save_path)
print("已保存至:", save_path.fpath)

Saved BaseData to /tmp/tmpcowinz_9/mne/NOD-MEG/sub-01_ses-ImageNet01_task-ImageNet_run-01.h5


已保存至: /tmp/tmpcowinz_9/mne/NOD-MEG/sub-01_ses-ImageNet01_task-ImageNet_run-01.h5


In [7]:
loaded: vtk.BaseData = vtk.read(save_path)
loaded

Loading VTK data from /tmp/tmpcowinz_9/mne/NOD-MEG/sub-01_ses-ImageNet01_task-ImageNet_run-01.h5


Loaded VTK data (lazy): neuro shape (80000, 273)


Time points,80000
Channels,273
Sampling frequency,250.00 Hz
Highpass,0.10 Hz
Lowpass,100.00 Hz
Data mode,continuous
n_visual,200
Baseline,"[-50, 0]"
Trial window,"[-50, 200]"


In [8]:
print("vision.db 类型:", type(loaded.vision.db))  # LazyH5Dict
first_key = list(loaded.vision.db.keys())[0]
img_arr = loaded.vision.db[first_key]
print(f"图片 {first_key!r} : shape={img_arr.shape}, dtype={img_arr.dtype}")

vision.db 类型: <class 'vneurotk.io.loader.LazyH5Dict'>
图片 'n01440764_26515' : shape=(375, 375, 3), dtype=uint8


---
## Part 2  Ephys 数据（MonkeyVision）

### 6. TrialRaster — 时间序列（spike raster）

`TrialRaster` 是 trial 级别的连续 spike 时序，`data_mode="epochs"`，
读取后已自动完成 `configure()`，无需再次调用。

In [9]:
mv_root = sample.data_path("monkey-vision")
EPHYS_ROOT = mv_root / "monkey-vision"
SES = sample.EPHYS_SESSION_ID

raster_path = EphysPath(root=EPHYS_ROOT, session_id=SES, dtype="TrialRaster")
print("Raster 文件:", raster_path.fpath)

Raster 文件: /var/tmp/zzl-zgh/cache/vneurotk/vneurotk-samples/monkey-vision/sessions/251024_FanFan_nsd1w_MSB/TrialRaster_251024_FanFan_nsd1w_MSB.h5


In [10]:
bd_raster = vtk.read(raster_path)
print(bd_raster)  # neuro=<lazy>
print(f"\nn_trials  = {bd_raster.n_trials}")
print(f"ntime     = {bd_raster.ntime}")
print(f"nchan     = {bd_raster.nchan}")
print(f"configured= {bd_raster.configured}")

Loaded ephys TrialRaster (lazy): 50932 trials, 350 timebins, 333 channels


BaseData(ntime=350, nchan=333, n_trials=50932, configured=True, data_mode='epochs', neuro=<lazy>)

n_trials  = 50932
ntime     = 350
nchan     = 333
configured= True


In [11]:
# trial_meta 来自 TrialRecord.csv，无需手动读取
print(bd_raster.trial_meta.head())
print(f"\ntrial_meta columns: {list(bd_raster.trial_meta.columns)}")

   id  start_time  stop_time  stim_index        stim_name  fix_success  \
0   0   19.382246  19.532246        5628  collect4500.png            1   
1   1   19.682250  19.832250        4693  collect3565.png            1   
2   2   19.982254  20.132254        4154  collect3026.png            1   
3   3   20.282257  20.432257        9673    share0672.png            1   
4   4   20.582261  20.732261        6551  collect5423.png            1   

   is_test  
0     True  
1     True  
2     True  
3     True  
4     True  

trial_meta columns: ['id', 'start_time', 'stop_time', 'stim_index', 'stim_name', 'fix_success', 'is_test']


In [12]:
bd_raster.load()
neuro = bd_raster.neuro
print(f"neuro shape: {neuro.shape}")  # (n_trials, n_timebins, n_units)

Lazy-loading neuro data...


Loading COO sparse data from /var/tmp/zzl-zgh/cache/vneurotk/vneurotk-samples/monkey-vision/sessions/251024_FanFan_nsd1w_MSB/TrialRaster_251024_FanFan_nsd1w_MSB.h5


neuro shape: (50932, 350, 333)


In [13]:
ephys_save = Path(tempfile.mkdtemp()) / "ephys_raster.h5"
bd_raster.save(ephys_save)
bd_back = vtk.read(ephys_save)
bd_back.load()
print(f"重加载: shape={bd_back.neuro.shape}, n_trials={bd_back.n_trials}")

Saved BaseData to /tmp/tmp2wsdyfu1/ephys_raster.h5


Loading VTK data from /tmp/tmp2wsdyfu1/ephys_raster.h5


Loaded VTK data (lazy): neuro shape (50932, 350, 333)


Lazy-loading neuro data...


Lazy-loading COO data from /tmp/tmp2wsdyfu1/ephys_raster.h5


重加载: shape=(50932, 350, 333), n_trials=50932


### 7. MeanFr / ChMeanFr — trial 级平均发放率

`data_mode="epochs"`，但无时间轴——shape 为 `(n_trials, n_units/n_channels)`。

In [14]:
bd_mfr = vtk.read(EphysPath(root=EPHYS_ROOT, session_id=SES, dtype="MeanFr"))
print(f"MeanFr   : data_mode={bd_mfr.data_mode}, shape={bd_mfr.neuro.shape}")

bd_cmfr = vtk.read(EphysPath(root=EPHYS_ROOT, session_id=SES, dtype="ChMeanFr"))
print(f"ChMeanFr : data_mode={bd_cmfr.data_mode}, shape={bd_cmfr.neuro.shape}")

Loaded ephys MeanFr (trial-level): shape (50932, 333)


MeanFr   : data_mode=patterns, shape=(50932, 333)


Loaded ephys ChMeanFr (trial-level): shape (50932, 384)


ChMeanFr : data_mode=patterns, shape=(50932, 384)


### 8. ChStimFr — stimulus 级发放率

`data_level="stimulus"`，每行对应一个唯一刺激，shape 为 `(n_stimuli, n_channels)`。
此类数据不支持 `configure()` 和 `crop()`。

In [15]:
bd_sfr = vtk.read(EphysPath(root=EPHYS_ROOT, session_id=SES, dtype="ChStimFr"))
print(bd_sfr)
print(f"data_mode={bd_sfr.data_mode}, shape={bd_sfr.neuro.shape}")

Loaded ephys ChStimFr (stimulus-level): shape (10060, 384)


BaseData(ntime=10060, nchan=384, n_trials=0, configured=False, data_mode='patterns')
data_mode=patterns, shape=(10060, 384)


In [16]:
bd_sfr.info

Time points,10060
Channels,384
Sampling frequency,N/A
Highpass,N/A
Lowpass,N/A
Data mode,patterns
Status,Not configured
Status,Not configured
